# 辅助视觉设备渲染管线（Notebook 版）

这个 Notebook 对应仓库中的渲染管线实现，适合在 Jupyter Notebook 中逐步运行与调参。

## 1. 环境准备
- Notebook 会自动向上查找项目根目录（包含 `src/biopiccw/pipeline.py`）。
- 无论你从仓库根目录还是 `notebooks/` 目录启动 Jupyter，均可导入 `biopiccw.pipeline`。

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'src' / 'biopiccw' / 'pipeline.py').exists():
            return candidate
    # 回退到当前目录（若用户在其它位置打开 notebook）
    return current

PROJECT_ROOT = find_project_root(Path.cwd())
SRC = PROJECT_ROOT / 'src'
for p in (PROJECT_ROOT, SRC):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print('Working directory:', Path.cwd())
print('Detected project root:', PROJECT_ROOT)
print('src in sys.path:', str(SRC) in sys.path)


## 2. 在 Notebook 内实现核心渲染函数（独立运行）
下面直接在 Notebook 中定义完整渲染管线函数，便于调试和理解，不依赖 `biopiccw.pipeline` 外部导入。

In [ ]:
import time
from pathlib import Path
import numpy as np
from skimage import exposure, filters, io, transform

# Step 1: 图像加载与输入（使用 skimage 读取并归一化到 [0,1]）
def load_image(image_path):
    image = io.imread(str(image_path))
    if image is None:
        raise FileNotFoundError(f'Cannot read image: {image_path}')
    if image.ndim == 2:
        image = np.stack([image, image, image], axis=-1)
    elif image.ndim == 3 and image.shape[2] == 4:
        image = image[:, :, :3]
    return image.astype('float32') / 255.0

# Step 2: 图像放大（统一用 skimage）
def apply_magnification(image, magnification_factor):
    if magnification_factor <= 0:
        raise ValueError('magnification_factor must be > 0')
    h, w = image.shape[:2]
    new_h = max(1, int(h * magnification_factor))
    new_w = max(1, int(w * magnification_factor))
    resized = transform.resize(image, (new_h, new_w, image.shape[2]), order=3, anti_aliasing=True, preserve_range=True)
    return np.clip(resized.astype('float32'), 0.0, 1.0)

# Step 3: 对比度增强（[0,1] 归一化域）
def enhance_contrast(image, contrast_factor):
    if contrast_factor < 0:
        raise ValueError('contrast_factor must be >= 0')
    enhanced = (image - 0.5) * contrast_factor + 0.5
    return np.clip(enhanced, 0.0, 1.0).astype('float32')

# Step 4: 边缘锐化（skimage）
def sharpen_image(image, sharpness_factor):
    if sharpness_factor < 0:
        raise ValueError('sharpness_factor must be >= 0')
    out = filters.unsharp_mask(image, radius=1.0, amount=sharpness_factor, preserve_range=True, channel_axis=-1)
    return np.clip(out, 0.0, 1.0).astype('float32')

# Step 5: 动态范围调整（Gamma，skimage）
def adjust_dynamic_range(image, gamma):
    if gamma <= 0:
        raise ValueError('gamma must be > 0')
    out = exposure.adjust_gamma(image, gamma=gamma)
    return np.clip(out, 0.0, 1.0).astype('float32')

# Step 6: 延迟模拟
def simulate_latency(image, delay_seconds):
    if delay_seconds < 0:
        raise ValueError('delay_seconds must be >= 0')
    time.sleep(delay_seconds)
    return image

# Step 7: 串联渲染管线
def render_pipeline(image_path, magnification_factor, contrast_factor, sharpness_factor, gamma, delay_seconds):
    image = load_image(image_path)
    image = apply_magnification(image, magnification_factor)
    image = enhance_contrast(image, contrast_factor)
    image = sharpen_image(image, sharpness_factor)
    image = adjust_dynamic_range(image, gamma)
    image = simulate_latency(image, delay_seconds)
    return image

# Step 8: 输出保存（使用 skimage.io.imsave）
def save_image(image, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    image_u8 = np.clip(image * 255.0, 0, 255).astype('uint8')
    io.imsave(str(output_path), image_u8)

print('Notebook core functions are ready (all skimage + normalized [0,1]).')


## 3. 直接读取你提供的测试图片路径
默认使用你提供的三通道 JPG：`D:\\vrcontent\\biopiccw\\test.jpg`。

In [ ]:
# 直接读取你提供的三通道 JPG 路径（skimage）
input_path = Path(r'D:\\vrcontent\\biopiccw\\test.jpg')
output_path = PROJECT_ROOT / 'notebooks' / 'user_test_output.jpg'

if not input_path.exists():
    raise FileNotFoundError(f'未找到测试图片: {input_path}。请在本机环境运行。')

img_ski = io.imread(str(input_path))
if img_ski is None:
    raise RuntimeError(f'skimage.io 无法读取图像: {input_path}')

print('skimage read shape:', getattr(img_ski, 'shape', None))
print('Input image path:', input_path)
print('Output image path:', output_path)


## 4. 设置参数并执行渲染

In [ ]:
magnification_factor = 2.0
contrast_factor = 1.5
sharpness_factor = 2.0
gamma = 2.2
delay_seconds = 0.0

output_image = render_pipeline(
    image_path=input_path,
    magnification_factor=magnification_factor,
    contrast_factor=contrast_factor,
    sharpness_factor=sharpness_factor,
    gamma=gamma,
    delay_seconds=delay_seconds,
)

save_image(output_image, output_path)
print('Rendered image saved to', output_path)
print('Output shape (H, W, C):', output_image.shape)


## 5. 查看结果（在支持的 Jupyter 环境中显示）

In [ ]:
# 如果你的环境安装了 IPython.display，可取消注释直接显示图像
# from IPython.display import display
# display(output_image)

print('Input path :', input_path)
print('Output path:', output_path)

## 6. 参数校验示例（可选）
下面示例展示非法参数会抛出 `ValueError`。

In [ ]:
try:
    _ = render_pipeline(
        image_path=input_path,
        magnification_factor=0,
        contrast_factor=1.0,
        sharpness_factor=1.0,
        gamma=1.0,
        delay_seconds=0.0,
    )
except ValueError as e:
    print('Caught expected error:', e)